# 3.2 Brand classification with transfer learning and angle information. 

In [1]:
import pandas as pd
import os
import sys
sys.path.append('../../utils')
import config_handling as conf
import cnn_helpers

from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import AdamW

2025-03-13 07:34:00.729739: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 3.2.1 Preparing the system
- Setting constants to us in this notebook and GPU override. 
- Reading config file - no database connection needed as data comes from CSVs

In [ ]:
SHAPE = 224   #required for resnet
BATCH_SIZE = 32     #how big ar teh batches for the online learning part
MAX_EPOCHS = 100    #upper limit of epochs per learning task - 
                    #    NOTE THAT there is early stopping and lr plateau just as in nobteook 2.
MODE = 'EXPERIMENTAL'       #EXPERIMENTAL (run tiny trainer) of FULL  (train all)

In [3]:
cnn_helpers.system_override()
device = cnn_helpers.system_pick_device()

System override applied - check if GPU is detected
Using GPU for deep learning.


2025-03-13 07:34:02.733624: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-03-13 07:34:04.985442: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-03-13 07:34:04.985525: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero


In [4]:
config = conf.read_config('../../config/automotive.conf.ini')
basedir = config['settings']['image_directory']

augment_base = os.path.join(basedir, 'augmentated data', 'brand phase')
augment_csv_dump = os.path.join(basedir, 'CSV-data', 'brand phase')


In [5]:
#local functions: 

def get_brands_by_augmented_data(df):
    """returns a sorted list of brandnames from least to most augmented data volume
    argument    
        df = dataframe of trainingsdata with abs_path and brand column required!
    """
    all_brands = df.brand.unique()
    filtered_df = df[df['abs_path'].str.contains('augmentated data', case=False, na=False)]
    brand_counts = filtered_df['brand'].value_counts()
    sorted_brands = brand_counts.index.tolist()
    for brand in all_brands:
        if brand not in sorted_brands: 
            sorted_brands.append(brand)
    sorted_brands = sorted_brands[::-1]
    #print(brand_counts)
    return sorted_brands

def interval_select(objects, n):
    """
        Returns a list of n-length with items taken out of objects evenly spaced. 
        eg. :   objects = [1,2,3,4,5,6,7,8,9]
                n=3
                will return 1,5,9
        returns are approximately with a skew at the end.; 
        e.g.    n = 4
                will return 1, 4, 6, 9 
    """
    if len(objects) < n:
        raise ValueError("The list must have at least 'n' elements.")
    idxs= [0]  #first index
    step = (len(objects) - 1) / (n - 1)
    for i in range(1, n - 1):
        idxs.append(round(i * step))
    idxs.append(len(objects) - 1) #last indes
    return [objects[i] for i in idxs]

def get_X_y(df): 
    """helper function for this notebook only as it's used a few times"""
    X = df.drop(columns=['brand', 'y_encoded'])
    y = df['y_encoded']
    return [X, y]

## 3.2.2 Data loading
Load the CSV's generated by step 3.1 into memory. 

If KERAS would've been able of using integers; then the 300.000 images per angle would've fit in memory: you'd need about 45GB; with the floats however you need about 4 times more! So here too you'll be required to use batch learning. 

In [6]:
traindata = pd.read_csv(os.path.join(augment_csv_dump, 'traindata_brandphase.csv'))
testdata = pd.read_csv(os.path.join(augment_csv_dump, 'testdata_brandphase.csv'))

In [7]:
angles = traindata['model_label'].unique()

## 3.2.3 Deciding on cropping or not

There's reason to assume that cropping according to the provided bounding boxes will yield the best results. We'll test this assumption using data for the front-angle view and 5 brands. These 5 brands will be chosen based on an even distribution of augmented data. i.e. we'll first calculate how many of the 10.000 records per brand are augmented, sort them and then select at even intervals. We'll be doing this small experiment here and apply the outcome of this in all `3.y`-notebooks. 

In [8]:
traindata.sample(3)

,image_id,model_label,model_score,yolobox_top_left_x,yolobox_top_left_y,yolobox_bottom_right_x,yolobox_bottom_right_y,bintag_predicts.image_id,model1_results,model2_results,model3_results,model4_results,brand,abs_path
69832,1622895,rearright,1.000000,112,101,685,457,1622895,0.998837,0.999990,0.989036,0.975101,alfa-romeo,/home/frederic/Documents/automotive_image_data...
66930,1614490,rearright,0.999999,140,195,1378,596,1614490,0.999228,1.000000,0.998229,0.997949,alfa-romeo,/home/frederic/Documents/automotive_image_data...
1747313,9547364,rearright,0.972284,28,177,722,482,9547364,0.999069,0.999998,0.998439,0.999010,porsche,/home/frederic/Documents/automotive_image_data...


For this mini-experiment, we'll do a small subselection of the data, this subselection will allow for quicker testing of the resnet learner that's planned on being used here. And will also allow testing whether or not cropping an image is beneficient or not. Just as in notebook `2`, there will be no extra model trained that applied a padded crop. 

We do this in a loop to allow us to extend the test if we're interested at some point. 

Just as you'd normally do, we'll be going through all the hoops of training: 
- Unlike notebook `2`, here we can use a simple labelencoder - there's no order we want to visualize later or adjacentness to keep in mind. We'll also shuffle the notebook and return to conventional Xy notation for df names.
- Notice that the data gets shuffled!! Up untill this point the traindata are 30 blocks of 80.000 images per brand!! 
- Class balances are taken care of in the previous step at notebook `3.1`; the same is true for the train-test split. 

In [ ]:
def resnet_learner(X_train, X_test, y_train, y_test, shape, apply_crop, storagefolder):
    train_gen = cnn_helpers.image_generator(
        batch_size=BATCH_SIZE, 
        data_frame=X_train, 
        bboxs=apply_crop, 
        shape=shape, 
        y_train_encoded=y_train
    )

    #Split validation separately
    val_gen = cnn_helpers.image_generator(
        batch_size=BATCH_SIZE, 
        data_frame=X_test, 
        bboxs=apply_crop, 
        shape=shape, 
        y_train_encoded=y_test
    )
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(shape, shape, 3))
    base_model.trainable = False

    model = models.Sequential([
        base_model,  # resnet50
        layers.GlobalAveragePooling2D(), 
        layers.Dense(256, activation='relu'),
        layers.Dense(y_train.nunique(), activation='softmax') 
    ])
    #sparse_categorical_crossentropy no need to use OHE with sparse_categorical_crossentropy!!!!
    model.compile(optimizer=AdamW(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    early_stopping = EarlyStopping(
        monitor='val_loss', 
        patience=6, 
        restore_best_weights=True
    )
    lr_scheduler = ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.5, 
        patience=3, 
        min_lr=1e-6, 
        verbose=1
    )
    model_checkpoint = ModelCheckpoint(
        os.path.join(storagefolder, 'disposable_dump_of_brand_model.keras'),
        'model_epoch_{epoch:02d}.h5',  # Save model with epoch number
        #monitor='val_loss',            # Save based on validation loss
        save_best_only=False,          # Save after every epoch; so I can resume from crash!!
        mode='min',
        save_weights_only=False,       # Save the full model, not just the weights
        verbose=0
    )

    model.fit(
        train_gen, 
        steps_per_epoch=len(X_train) // BATCH_SIZE,  # Number of batches per epoch
        epochs=MAX_EPOCHS,
        validation_data=val_gen, 
        validation_steps=len(X_test) // BATCH_SIZE,  # Number of validation batches per epoch
        callbacks=[lr_scheduler, early_stopping, model_checkpoint]  # Use the learning rate scheduler callback
    )

    #the second .fit() method is with unfrozen base: this should be more precise for fine-tuning
    base_model.trainable = True
    model.compile(optimizer=AdamW(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(
        train_gen, 
        steps_per_epoch=len(X_train) // BATCH_SIZE, 
        epochs=MAX_EPOCHS,
        validation_data=val_gen,
        validation_steps=len(X_test) // BATCH_SIZE,
        callbacks=[lr_scheduler, early_stopping, model_checkpoint]
    )
    
    return model


Epoch 1/100


I0000 00:00:1741847760.786643   26590 service.cc:146] XLA service 0x7466bc04d950 initialized for platform ROCM (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1741847760.786683   26590 service.cc:154]   StreamExecutor device (0): AMD Radeon RX 6700 XT, AMDGPU ISA version: gfx1030
2025-03-13 07:36:00.874708: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


   1/1562 ━━━━━━━━━━━━━━━━━━━━ 3:08:51 7s/step - accuracy: 0.2812 - loss: 1.5936

I0000 00:00:1741847764.769211   26590 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1562/1562 ━━━━━━━━━━━━━━━━━━━━ 237s 147ms/step - accuracy: 0.4394 - loss: 1.3540 - val_accuracy: 0.7075 - val_loss: 0.8118 - learning_rate: 0.0010
Epoch 2/100
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 228s 146ms/step - accuracy: 0.6568 - loss: 0.9224 - val_accuracy: 0.7311 - val_loss: 0.7447 - learning_rate: 0.0010
Epoch 3/100
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 228s 146ms/step - accuracy: 0.7012 - loss: 0.7995 - val_accuracy: 0.6549 - val_loss: 0.8892 - learning_rate: 0.0010
Epoch 4/100
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 227s 145ms/step - accuracy: 0.7284 - loss: 0.7307 - val_accuracy: 0.6636 - val_loss: 0.8695 - learning_rate: 0.0010
Epoch 5/100
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 227s 146ms/step - accuracy: 0.7357 - loss: 0.6967 - val_accuracy: 0.7455 - val_loss: 0.6629 - learning_rate: 0.0010
Epoch 6/100
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 228s 146ms/step - accuracy: 0.7568 - loss: 0.6462 - val_accuracy: 0.7594 - val_loss: 0.6371 - learning_rate: 0.0010
Epoch 7/100
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 230s 147ms/step

In [ ]:
if MODE != 'EXPERIMENTAL':
    raise Exception('SKIPPING TINY MODEL TRAINING')


experiment_values = [True, False]       #values to test for crop argument.
brands_in_experiment = 5                #how many brands in this experiment
chosen_angle_croptest = ['front']       #if we ever want to test multiple angles, this can be extended

model_dest_dir = os.path.join(os.getcwd(), '..', '..', 'models', 'brand_models')
os.makedirs(model_dest_dir, exist_ok=True)

for angle in chosen_angle_croptest: 
    view_by_angle = traindata.query('model_label==@angle')
    brands_desc = get_brands_by_augmented_data(view_by_angle)
    selected_brands = interval_select(brands_desc, brands_in_experiment)
    small_df_train = view_by_angle[view_by_angle['brand'].isin(selected_brands)].copy()
    small_df_test = testdata.query('model_label==@angle')
    small_df_test = small_df_test[small_df_test['brand'].isin(selected_brands)].copy()
    #Shuffle train and test set. 
    small_df_train = cnn_helpers.shuffle_df(small_df_train)
    small_df_test = cnn_helpers.shuffle_df(small_df_test)
    # apply label encoding: we need to redo this for every iteration of the loop 
    #   as you can't be sure selected_brands is always the same!!
    tiny_label_encoder = LabelEncoder()
    tiny_label_encoder.fit(selected_brands)
    small_df_train['y_encoded'] = tiny_label_encoder.transform(small_df_train['brand'])
    small_df_test['y_encoded'] = tiny_label_encoder.transform(small_df_test['brand'])
    # Conventional naming schemes: Xy-notatation for train and test 
    #   rememember that traintestsplit is already done and was frozen
    #   because of the CSV dump. 
    X_train, y_train = get_X_y(small_df_train)
    X_test, y_test = get_X_y(small_df_test)
    for crop in experiment_values:
        name = f'TINY_brand_model-Resnet50_cropped={crop}.keras'
        model = resnet_learner(X_train, X_test, y_train, y_test, SHAPE, crop, model_dest_dir)
        model.save(os.path.join(model_dest_dir, name))



In [13]:
os.listdir(model_dest_dir)

['disposable_dump_of_brand_model.keras',
 'TINY_brand_model-Resnet50_cropped=False.keras',
 'TINY_brand_model-Resnet50_cropped=True.keras']

In [ ]:
if MODE != 'EXPERIMENTAL':
    raise Exception('SKIPPING TINY MODEL TRAINING')

#now that tiny models are trained: evaluate the performance. 
#TODO tomorrow